In [5]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt 

In [6]:
from pathlib import Path

# Define the project root relative to this notebook's location
#BASE_DIR = Path(__file__).resolve().parent.parent  # if using a .py script
# OR for a Jupyter notebook (most reliable):
BASE_DIR = Path.cwd().parent  # assumes notebook is in /Notebooks/

CLEANED = BASE_DIR / "Data" / "cleaned"

# Load the datasets
npc_df = pd.read_csv(CLEANED / "amac_rental_cleaned_v2.csv")
ppro_df = pd.read_csv(CLEANED / "amac_ppro_rental_cleaned_v2.csv")

In [ ]:


# Step 1: Add a 'Source' column (Best practice for tracking data lineage)
npc_df['Source'] = 'Nigeria Property Centre'
ppro_df['Source'] = 'Property Pro NG'

# Step 2: Combine the datasets vertically
df = pd.concat([npc_df, ppro_df], axis=0, ignore_index=True)

# Save the master dataset for your Streamlit App
df.to_csv(CLEANED / "amac_rental_master.csv", index=False)
df.shape

(5859, 5)

In [3]:
# Group by the key features AND the Source
df_refined = df.drop_duplicates(subset=['Bedrooms', 'Price(Float)', 'District', 'Property Category', 'Source'])

print(f"Refined count: {len(df_refined)} listings")
# Check if a district got wiped out
print(df_refined['District'].value_counts().head(10))

Refined count: 2435 listings
District
Wuse         238
Maitama      222
Asokoro      196
Gwarinpa     181
Guzape       163
Jahi         158
Katampe      153
Lugbe        144
Life Camp    140
Jabi         126
Name: count, dtype: int64


In [4]:
# Descriptive Statistics
print(df.describe())

# Check for the "Abuja Skew"
print(f"Median Rent: ₦{df['Price(Float)'].median():,.2f}")
print(f"Mean Rent: ₦{df['Price(Float)'].mean():,.2f}")

          Bedrooms  Price(Float)
count  5444.000000  5.859000e+03
mean      5.170096  3.472020e+08
std      28.973904  2.302697e+10
min       1.000000  1.300000e+02
25%       3.000000  7.000000e+06
50%       3.000000  1.300000e+07
75%       4.000000  2.500000e+07
max    1150.000000  1.761461e+12
Median Rent: ₦13,000,000.00
Mean Rent: ₦347,201,961.63


In [5]:
# Create a 'Price per Bedroom' metric for fairer comparison
df['Price_per_Bed'] = df['Price(Float)'] / df['Bedrooms']


In [15]:
df.shape

(4967, 6)

In [7]:
#applying bedroom cap at 6 bedrooms to remove outliers
df = df[df["Bedrooms"] <= 6]
df["Bedrooms"].value_counts()

Bedrooms
4.0    1774
3.0    1577
2.0     874
5.0     555
1.0     308
6.0     197
Name: count, dtype: int64

In [24]:
#applying price cap at ₦50,000,000.00 to remove outliers
df = df[df["Price(Float)"] <= 50000000]
df = df[df['Price(Float)'] >= 250000]
df.tail()

,Bedrooms,Property Category,Price(Float),District,Source,Price_per_Bed
5850,4.0,Duplex,12000000.0,Gwarinpa,Property Pro NG,3000000.0
5853,1.0,Apartment,3000000.0,Mabushi,Property Pro NG,3000000.0
5855,3.0,Apartment,6000000.0,Katampe,Property Pro NG,2000000.0
5856,2.0,Apartment,3700000.0,Garki,Property Pro NG,1850000.0
5857,2.0,Apartment,3700000.0,Garki,Property Pro NG,1850000.0


In [9]:
df.dtypes

Bedrooms             float64
Property Category     object
Price(Float)         float64
District              object
Source                object
Price_per_Bed        float64
dtype: object

In [10]:
df.describe()

,Bedrooms,Price(Float),Price_per_Bed
count,4967.000000,4.967000e+03,4.967000e+03
mean,3.324945,1.568496e+07,4.692019e+06
std,1.140152,1.122096e+07,3.154279e+06
min,1.000000,7.500000e+04,4.250000e+04
25%,3.000000,7.000000e+06,2.500000e+06
50%,3.000000,1.200000e+07,3.750000e+06
75%,4.000000,2.000000e+07,6.000000e+06
max,6.000000,5.000000e+07,3.200000e+07


In [22]:
EMPTY = df.isnull().any(axis=1).sum()
print(EMPTY)

40


In [25]:
df = df.dropna()
print(df.isnull().any().sum())
print(df.shape)

0
(4920, 6)


In [28]:
#District Tiers (Budget, Mid-range, Luxury)
#Calculate the median price for every district
district_medians = df.groupby("District")["Price(Float)"].median()

#Define thresholds using Quantiles. 0-33% = Budget, 33-66% = Mid-Range, 66-100% = Luxury
tiers = pd.qcut(district_medians, q=3, labels=["Budget", "Mid-Range", "Luxury"])

df["District Tier"] = df["District"].map(tiers)


In [29]:
print(df[["District", "District Tier"]].head())
print(df[["District", "District Tier"]].tail())

  District District Tier
0     Jabi        Luxury
1    Kaura        Budget
2      Apo     Mid-Range
3      Apo     Mid-Range
4      Apo     Mid-Range
      District District Tier
5850  Gwarinpa        Budget
5853   Mabushi        Luxury
5855   Katampe        Luxury
5856     Garki     Mid-Range
5857     Garki     Mid-Range


In [30]:
#What is the median price per district (to avoid skew from outliers)?
district_medians = district_medians.sort_values()
# each median with ₦ and commas 
district_medians2 = district_medians.apply(lambda x: f"₦{x:,.2f}") 
print(district_medians2)

District
Karshi        ₦1,100,000.00
Lugbe         ₦4,500,000.00
Lokogoma      ₦5,000,000.00
Durumi        ₦6,000,000.00
Kaura         ₦6,000,000.00
Kabusa        ₦7,000,000.00
Idu           ₦7,000,000.00
Gwarinpa      ₦7,000,000.00
Life Camp     ₦7,000,000.00
Garki         ₦9,000,000.00
Kado         ₦10,000,000.00
Jahi         ₦10,000,000.00
Utako        ₦10,000,000.00
Apo          ₦10,000,000.00
Mabushi      ₦12,000,000.00
Guzape       ₦15,000,000.00
Wuye         ₦15,000,000.00
Katampe      ₦18,000,000.00
Wuse         ₦20,000,000.00
Asokoro      ₦22,500,000.00
Maitama      ₦25,000,000.00
Jabi         ₦32,000,000.00
Name: Price(Float), dtype: object


In [31]:
#Which district has the cheapest average rent in AMAC?
district_avg = df.groupby("District")["Price(Float)"].mean()
print(f"Cheapest District: {district_avg.idxmin()}")

Cheapest District: Karshi


In [32]:
#Which district has the most expensive average rent?
print(f"Most Expensive District in AMAC: {district_avg.idxmax()}")

Most Expensive District in AMAC: Maitama


In [33]:
#Which district offers the most affordable 1 bedroom apartment in AMAC?
one_bed = df[df["Bedrooms"] == 1]
cheapest_1bed_loc = one_bed.groupby("District")["Price(Float)"].mean().idxmin()
print(f"Best for 1 Beds: {cheapest_1bed_loc}")

Best for 1 Beds: Lokogoma


In [34]:
#Which district has the highest number of listings?
highest_district = df["District"].value_counts().idxmax()
print(f"District with the highest number of listings: {highest_district}")

District with the highest number of listings: Katampe


In [38]:
df.head()

,Bedrooms,Property Category,Price(Float),District,Source,Price_per_Bed,district_tier,District Tier
0,3.0,Duplex,32000000.0,Jabi,Nigeria Property Centre,1.066667e+07,Luxury,Luxury
1,3.0,Apartment,8500000.0,Kaura,Nigeria Property Centre,2.833333e+06,Budget,Budget
2,1.0,Apartment,1500000.0,Apo,Nigeria Property Centre,1.500000e+06,Mid-Range,Mid-Range
3,3.0,Apartment,10000000.0,Apo,Nigeria Property Centre,3.333333e+06,Mid-Range,Mid-Range
4,2.0,Apartment,7500000.0,Apo,Nigeria Property Centre,3.750000e+06,Mid-Range,Mid-Range


In [40]:
df_new = df.drop(['Price_per_Bed','district_tier'], axis =1)
df_new.head()

,Bedrooms,Property Category,Price(Float),District,Source,District Tier
0,3.0,Duplex,32000000.0,Jabi,Nigeria Property Centre,Luxury
1,3.0,Apartment,8500000.0,Kaura,Nigeria Property Centre,Budget
2,1.0,Apartment,1500000.0,Apo,Nigeria Property Centre,Mid-Range
3,3.0,Apartment,10000000.0,Apo,Nigeria Property Centre,Mid-Range
4,2.0,Apartment,7500000.0,Apo,Nigeria Property Centre,Mid-Range


In [43]:
df_new.dtypes

Bedrooms              float64
Property Category      object
Price(Float)          float64
District               object
Source                 object
District Tier        category
dtype: object

In [ ]:

#Impovement 2
df.to_csv(CLEANED / "amac_rental_master_v2.csv", index=False)